# DuckDB: SQL analytics and ML feature engineering

A practical, beginner-to-advanced introduction with an end-to-end **online retail analytics project**.
Read the explanations, run cells in order, and inspect each result. Additional explanations and troubleshooting are included in the reference sections below.

**You will learn:** connections; SQL; types and NULLs; CSV/JSON/Parquet; DataFrames and Arrow; joins; CTEs; windows; quality checks; transactions; query plans; persistence; and leakage-aware ML features.

**Project questions:** How does completed-sales revenue change over time? Which products and regions contribute most? How many orders are returned? Which customers are active or at risk? Can historical purchase features predict a purchase next month?

**Prerequisites:** basic Python and an understanding of rows and columns. SQL is introduced here. Allow 2–3 hours to study, although execution is much shorter.

**Dataset:** deterministic, synthetic data generated below: 1,200 customers, 60 products, 24,000 orders across calendar 2024, plus 80 exact duplicate records. This is fictional teaching data, not evidence about actual shoppers. One row is one single-product order, not a multi-line basket. Amounts are fictional USD; taxes, shipping, partial returns and acquisition costs are excluded.

**How to study:** before running a query, predict the result’s grain, define the denominator, and state what evidence would change your interpretation. After running it, check an invariant and explain one limitation. The sections start with situations and analytical decisions rather than isolated API lists.

## 1. Setup and execution
Use the repository's Python environment, or install this smaller set in your notebook kernel:
```python
%pip install duckdb==1.4.3 numpy==1.26.4 pandas==2.3.3 matplotlib==3.9.4 pyarrow==21.0.0 scikit-learn==1.6.1
```
The pin targets this repository's Python 3.9 environment; it is a reproducibility pin, not a claim that this is the newest release. Restart the kernel after installation.

Files generated by this notebook live under `outputs/duckdb/`. Re-running from the beginning replaces its generated inputs and reports and recreates its named database tables. No network or external dataset is required after dependencies are installed.

In [ ]:
from pathlib import Path
import json
import platform
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

OUT = Path("outputs/duckdb")
OUT.mkdir(parents=True, exist_ok=True)
print({"python": platform.python_version(), "duckdb": duckdb.__version__,
       "numpy": np.__version__, "pandas": pd.__version__})
# Close a previous connection when rerunning this setup cell.
if "con" in globals():
    con.close()
con = duckdb.connect(str(OUT / "retail.duckdb"))
con.execute("SET threads = 2")
con.execute("SET memory_limit = '512MB'")
# This limit does not cap the total Python process's memory.

## 2. A sales question spans more rows than you want to inspect by hand

Imagine receiving a year of orders and being asked for sales totals by region. Opening each row is unnecessary: you need an engine that can scan columns and summarize them. Start by checking a tiny expression so you can distinguish Python execution from SQL execution.
DuckDB is an embedded analytical SQL database: it runs inside Python without a separate database server. It is useful for scanning, joining and aggregating tables and files. A file-backed connection persists tables; `duckdb.connect()` creates a temporary in-memory database.

`execute()` runs SQL; `fetchone()` returns one tuple; `fetchall()` returns tuples; `fetchdf()` creates a pandas DataFrame. Fetch only the data you need: converting a huge result into pandas still requires memory.

The query below introduces aliases (`AS`), explicit types (`CAST`), and arithmetic. DuckDB SQL is similar to PostgreSQL, but is not identical.

In [ ]:
display(con.execute("""
    SELECT 'Hello DuckDB' AS message,
           6 * 7 AS answer,
           CAST(19.95 AS DECIMAL(10,2)) AS price,
           DATE '2024-01-01' + INTERVAL '7 days' AS next_week
""").fetchdf())
print(con.execute("SELECT 42").fetchone())

## 3. Decide what a row means before calculating anything

The store manager asks which products deserve attention. Before adding prices, ask whether a row represents an order, an item, or a payment. Counting item rows as orders would overstate order frequency. Here we deliberately choose one single-product order per row, so COUNT(*) can represent orders after deduplication.
The generator uses a fixed NumPy seed. Customer purchasing propensity varies; channel and region are assigned fictional distributions. There is no claim that this represents realistic causal behavior or a production ML benchmark.

**Data dictionary**

| Table | Key / grain | Other columns |
|---|---|---|
| customers | `customer_id`, one customer | region, signup_date |
| products | `product_id`, one product | category, unit_cost (USD per unit) |
| orders | `order_id`, one order | customer_id, product_id, order_date, quantity, unit_price, discount, status, channel |

`discount` is a fraction from 0 to 0.25. `unit_price` is the price before discount. Status is completed, returned, or cancelled and is treated as known on the order date for this simplified project. Real historical status changes require event timestamps or historical snapshots.

We inject 80 duplicates, 40 invalid prices, 30 invalid quantities, and 60 missing channels. These groups do not overlap. Missing channels will be retained as `unknown`; invalid quantities/prices will be quarantined. Missing dates, invalid discounts, unknown statuses and orphan keys are also checked.

In [ ]:
rng = np.random.default_rng(42)
customers = pd.DataFrame({
    "customer_id": np.arange(1, 1201),
    "region": rng.choice(["North", "South", "East", "West"], 1200),
    "signup_date": pd.Timestamp("2023-01-01") + pd.to_timedelta(rng.integers(0, 365, 1200), unit="D")
})
products = pd.DataFrame({
    "product_id": np.arange(1, 61),
    "category": np.repeat(["Electronics", "Home", "Sports", "Books", "Clothing"], 12),
    "unit_cost": np.round(rng.uniform(4, 80, 60), 2)
})
propensity = rng.gamma(shape=1.3, scale=1.0, size=1200)
product_ids = rng.integers(1, 61, 24000)
orders = pd.DataFrame({
    "order_id": np.arange(1, 24001),
    "customer_id": rng.choice(customers.customer_id, 24000, p=propensity / propensity.sum()),
    "product_id": product_ids,
    "order_date": pd.Timestamp("2024-01-01") + pd.to_timedelta(rng.integers(0, 366, 24000), unit="D"),
    "quantity": rng.integers(1, 6, 24000),
    "unit_price": np.round(products.unit_cost.to_numpy()[product_ids - 1] * rng.uniform(1.2, 2.0, 24000), 2),
    "discount": rng.choice([0.0, 0.05, 0.10, 0.25], 24000),
    "status": rng.choice(["completed", "returned", "cancelled"], 24000, p=[0.87, 0.08, 0.05]),
    "channel": rng.choice(["web", "app", "store"], 24000)
})
orders.loc[0:39, "unit_price"] = -1
orders.loc[40:69, "quantity"] = 0
orders.loc[70:129, "channel"] = None
raw_orders = pd.concat([orders, orders.iloc[200:280]], ignore_index=True)
customers.to_csv(OUT / "customers.csv", index=False)
products.to_parquet(OUT / "products.parquet", index=False)
raw_orders.to_csv(OUT / "orders.csv", index=False)
(OUT / "events.jsonl").write_text(
    '\n'.join(json.dumps(x) for x in [
        {"customer_id": 1, "event": "view", "tags": ["home", "sale"]},
        {"customer_id": 2, "event": "cart", "tags": ["books"]}
    ]) + '\n'
)
display(raw_orders.head())
print(f"{len(customers):,} customers, {len(products)} products, {len(raw_orders):,} raw orders")

## 4. Inspect unfamiliar files before building a database

An analyst receives customer CSV, product Parquet, and event JSON files. First inspect a few rows and their structure. Do not assume that every file needs to become a pandas DataFrame. Notice that unnesting event tags creates multiple rows per event; counting those rows would count tags, not events.
`read_csv` and `read_parquet` are table functions. DuckDB can query them directly. CSV type inference is convenient for exploration, but production ingestion should define types explicitly. Parquet already stores a schema.

Use `?` placeholders for values, including supported file-path parameters. They are not a mechanism for substituting SQL identifiers such as table names. Never concatenate untrusted input into SQL.

In [ ]:
display(con.execute("SELECT * FROM read_csv(?) LIMIT 5", [str(OUT / "customers.csv")]).fetchdf())
display(con.execute("SELECT category, COUNT(*) AS products FROM read_parquet(?) GROUP BY category ORDER BY category",
                    [str(OUT / "products.parquet")]).fetchdf())
display(con.execute("SELECT customer_id, event, unnest(tags) AS tag FROM read_json_auto(?)",
                    [str(OUT / "events.jsonl")]).fetchdf())

## 5. Preserve the evidence when an input file is messy

A price field may contain invalid text tomorrow. If ingestion silently changes or drops it, you lose the explanation for a suspicious total. Keep the raw values, create a typed layer separately, and inspect the schema before making business calculations.
A **table** stores data in the database. A **view** stores a query and computes results when read. A **CTE** names an intermediate query inside one SQL statement. A **registered DataFrame** is a connection-local way to query a Python object; it does not persist that object to disk.

We read raw orders as text to preserve malformed values. `TRY_CAST` later returns NULL for conversion failures, allowing us to audit them instead of aborting the entire load.

In [ ]:
con.execute("CREATE OR REPLACE TABLE customers AS SELECT * FROM read_csv(?)", [str(OUT / "customers.csv")])
con.execute("CREATE OR REPLACE TABLE products AS SELECT product_id, category, CAST(unit_cost AS DECIMAL(12,2)) AS unit_cost FROM read_parquet(?)", [str(OUT / "products.parquet")])
con.execute("CREATE OR REPLACE TABLE raw_orders AS SELECT * FROM read_csv(?, all_varchar=true)", [str(OUT / "orders.csv")])
display(con.execute("SHOW TABLES").fetchdf())
display(con.execute("DESCRIBE raw_orders").fetchdf())
display(con.execute("SUMMARIZE products").fetchdf())

## 6. Decide which records you can trust

Some orders appear twice, some have impossible prices, and some lack a channel. These issues need different decisions: remove exact duplicates, quarantine invalid financial records, and preserve valid orders with an unknown channel. Ask how each decision changes both row counts and revenue.
Money uses `DECIMAL`, avoiding binary floating-point storage for financial amounts. Revenue is rounded per order to two decimal places; cost is the product's fixed teaching cost. Some operations such as division and conversion to pandas may produce floating point results.

NULL means unknown, not zero: use `IS NULL`, not `= NULL`. `COUNT(*)` counts rows; `COUNT(column)` excludes NULLs. `COALESCE` fills a missing value; `NULLIF(x, 0)` protects denominators. `WHERE` keeps only true predicates, so an unknown predicate needs explicit handling.

Here exact duplicate rows are removed with `DISTINCT`. In a real system, conflicting versions of the same order need a documented timestamp/version rule, not arbitrary deduplication.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE typed_orders AS
SELECT TRY_CAST(order_id AS INTEGER) AS order_id,
       TRY_CAST(customer_id AS INTEGER) AS customer_id,
       TRY_CAST(product_id AS INTEGER) AS product_id,
       TRY_CAST(order_date AS DATE) AS order_date,
       TRY_CAST(quantity AS INTEGER) AS quantity,
       TRY_CAST(unit_price AS DECIMAL(12,2)) AS unit_price,
       TRY_CAST(discount AS DECIMAL(5,4)) AS discount,
       lower(trim(status)) AS status,
       COALESCE(NULLIF(lower(trim(channel)), ''), 'unknown') AS channel
FROM (SELECT DISTINCT * FROM raw_orders)
""")
con.execute("""
CREATE OR REPLACE TABLE audited_orders AS
SELECT *, CASE
    WHEN order_id IS NULL OR customer_id IS NULL OR product_id IS NULL THEN 'missing_key'
    WHEN order_date IS NULL OR order_date < DATE '2024-01-01' OR order_date >= DATE '2025-01-01' THEN 'invalid_date'
    WHEN unit_price IS NULL OR unit_price <= 0 THEN 'invalid_price'
    WHEN quantity IS NULL OR quantity <= 0 THEN 'invalid_quantity'
    WHEN discount IS NULL OR discount NOT BETWEEN 0 AND 1 THEN 'invalid_discount'
    WHEN status IS NULL OR status NOT IN ('completed', 'returned', 'cancelled') THEN 'invalid_status'
    WHEN customer_id NOT IN (SELECT customer_id FROM customers) THEN 'unknown_customer'
    WHEN product_id NOT IN (SELECT product_id FROM products) THEN 'unknown_product'
    ELSE NULL END AS rejection_reason
FROM typed_orders
""")
con.execute("CREATE OR REPLACE TABLE rejected_orders AS SELECT * FROM audited_orders WHERE rejection_reason IS NOT NULL")
con.execute("CREATE OR REPLACE TABLE clean_orders AS SELECT * EXCLUDE (rejection_reason) FROM audited_orders WHERE rejection_reason IS NULL")
display(con.execute("SELECT COALESCE(rejection_reason, 'accepted') AS outcome, COUNT(*) AS rows FROM audited_orders GROUP BY outcome ORDER BY outcome").fetchdf())

## 7. Prove that cleaning did not silently lose orders

A clean-looking table is not proof of a correct pipeline. Reconcile the original count against duplicates, accepted records, and rejected records. Then check uniqueness and foreign keys. Predict the accepted count before running the cell; if it differs, investigate rather than changing the assertion to fit.
These checks verify uniqueness, row reconciliation, and referential integrity before business queries. Expected outcomes: **24,080 raw → 24,000 deduplicated → 23,930 accepted + 70 rejected**. Rejections retain the first matching reason, rather than every possible issue. The raw table remains available for investigations.

In [ ]:
def scalar(sql):
    return con.execute(sql).fetchone()[0]

assert scalar("SELECT COUNT(*) FROM raw_orders") == 24080
assert scalar("SELECT COUNT(*) FROM typed_orders") == 24000
assert scalar("SELECT COUNT(*) FROM rejected_orders") == 70
assert scalar("SELECT COUNT(*) FROM clean_orders") == 23930
assert scalar("SELECT COUNT(*) - COUNT(DISTINCT order_id) FROM clean_orders") == 0
assert scalar("SELECT COUNT(*) - COUNT(DISTINCT customer_id) FROM customers") == 0
assert scalar("SELECT COUNT(*) - COUNT(DISTINCT product_id) FROM products") == 0
assert scalar("SELECT COUNT(*) FROM clean_orders o ANTI JOIN customers c USING(customer_id)") == 0
assert scalar("SELECT COUNT(*) FROM clean_orders o ANTI JOIN products p USING(product_id)") == 0
assert scalar("SELECT COUNT(*) FROM clean_orders WHERE channel='unknown'") == 60
print("Data contract passed.")

## 8. Compare channels over a clearly defined period

The manager wants to compare active channels in the second half of the year. Restrict the dates and status before aggregating; then exclude groups with too few orders. Ask whether a channel has more orders because it performs better or simply because it has more traffic—we have no visitor denominator here.
SQL's useful conceptual order is FROM/JOIN → WHERE → GROUP BY → HAVING → window calculations → QUALIFY → SELECT/DISTINCT → ORDER BY → LIMIT (the optimizer can execute differently).

`WHERE` filters individual orders; `HAVING` filters aggregate groups. Our parameterized query selects a minimum order date and includes channels with at least 100 completed orders. Explicit sorting makes displayed results deterministic.

In [ ]:
display(con.execute("""
SELECT channel, COUNT(*) AS completed_orders, SUM(quantity) AS units,
       ROUND(AVG(quantity), 2) AS average_units
FROM clean_orders
WHERE status = ? AND order_date >= ?
GROUP BY channel
HAVING COUNT(*) >= ?
ORDER BY completed_orders DESC, channel
""", ['completed', '2024-07-01', 100]).fetchdf())

## 9. Agree on what revenue means before presenting a total

Two analysts disagree because one includes cancelled orders and the other excludes them. Write a shared definition, then enrich orders with region and category. Check the row count before and after joining: unexpected growth often means a dimension key is duplicated.
A join can accidentally multiply rows when its keys are not unique; the checks above protect the many-to-one dimension joins.

We define **completed-sales revenue** as discounted order value for completed orders only. Returned/cancelled orders contribute zero revenue and zero product cost in this simplified ledger. Thus this is not a full accounting model or a historical cash-flow report. **Gross profit** is revenue minus product cost; it is not net profit. The return rate later uses returned / (completed + returned), excluding cancellations.

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW order_facts AS
SELECT o.*, c.region, p.category,
       CAST(ROUND(quantity * unit_price * (1 - discount), 2) AS DECIMAL(18,2)) AS order_value,
       CASE WHEN status='completed'
            THEN CAST(ROUND(quantity * unit_price * (1-discount), 2) AS DECIMAL(18,2))
            ELSE 0 END AS revenue,
       CASE WHEN status='completed' THEN quantity * p.unit_cost ELSE 0 END AS product_cost
FROM clean_orders o
JOIN customers c USING(customer_id)
JOIN products p USING(product_id)
""")
assert scalar("SELECT COUNT(*) FROM order_facts") == scalar("SELECT COUNT(*) FROM clean_orders")
display(con.execute("""
SELECT COUNT(*) AS accepted_orders,
       COUNT(*) FILTER (WHERE status='completed') AS completed_orders,
       SUM(revenue) AS revenue_usd,
       SUM(revenue-product_cost) AS gross_profit_usd,
       SUM(revenue) / NULLIF(COUNT(*) FILTER (WHERE status='completed'), 0) AS average_completed_order_usd,
       100.0 * COUNT(*) FILTER (WHERE status='returned') /
         NULLIF(COUNT(*) FILTER (WHERE status IN ('completed','returned')), 0) AS return_rate_pct
FROM order_facts
""").fetchdf())

## 10. Separate an actual revenue change from a missing month

A chart jumps from January to March. Is February zero, absent, or not yet loaded? A complete calendar makes that gap visible. Here the generated year is fully observed, so a missing sales month can be filled with zero. In real data, verify ingestion completeness before making that assumption. Inspect month-over-month growth alongside absolute revenue.
A CTE improves readability. Window functions retain rows while calculating across related rows. `LAG` retrieves the previous month; a windowed `SUM` computes cumulative revenue.

A calendar spine ensures missing months appear as zero rather than disappearing. Without it, “previous row” might mean several months ago. A moving window of three rows means three calendar months only because this spine fills every month.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE monthly_report AS
WITH calendar AS (
    SELECT CAST(month AS DATE) AS month
    FROM generate_series(DATE '2024-01-01', DATE '2024-12-01', INTERVAL '1 month') AS t(month)
), totals AS (
    SELECT CAST(date_trunc('month', order_date) AS DATE) AS month,
           SUM(revenue) AS revenue,
           SUM(revenue-product_cost) AS gross_profit
    FROM order_facts GROUP BY 1
), filled AS (
    SELECT c.month, COALESCE(t.revenue, 0) AS revenue,
           COALESCE(t.gross_profit, 0) AS gross_profit
    FROM calendar c LEFT JOIN totals t USING(month)
)
SELECT *, LAG(revenue) OVER (ORDER BY month) AS previous_month,
       100.0 * (revenue / NULLIF(LAG(revenue) OVER (ORDER BY month), 0) - 1) AS growth_pct,
       SUM(revenue) OVER (ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS cumulative_revenue,
       AVG(revenue) OVER (ORDER BY month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS moving_average_3m
FROM filled
""")
monthly = con.execute("SELECT * FROM monthly_report ORDER BY month").fetchdf()
assert len(monthly) == 12
assert scalar("SELECT SUM(revenue) FROM monthly_report") == scalar("SELECT SUM(revenue) FROM order_facts")
display(monthly.round(2))
ax = monthly.plot(x="month", y=["revenue", "moving_average_3m"], figsize=(10,4), marker="o")
ax.set(title="Completed-sales revenue: synthetic store", ylabel="USD", xlabel="Month")
plt.tight_layout()
plt.savefig(OUT / "monthly_revenue.png", dpi=140)
plt.show()

## 11. Find category leaders without letting the largest category dominate

A global top-three list may contain only electronics. Rank within each category to answer a different question: which products lead their own category? Decide whether ties should share a rank before writing the query. Revenue leaders may still have poor margins or high returns.
`ROW_NUMBER` provides an exact top three per category; product_id breaks revenue ties reproducibly. `DENSE_RANK` would instead preserve ties and might return more than three products. `QUALIFY` filters after window calculations.

`PIVOT` turns category values into columns. Keep a long table for most pipelines; use wide reports when they help readers compare values.

In [ ]:
display(con.execute("""
WITH product_sales AS (
    SELECT category, product_id, SUM(revenue) AS revenue
    FROM order_facts GROUP BY category, product_id
)
SELECT *, ROW_NUMBER() OVER (PARTITION BY category ORDER BY revenue DESC, product_id) AS position
FROM product_sales QUALIFY position <= 3
ORDER BY category, position
""").fetchdf())
display(con.execute("""
PIVOT (SELECT region, category, revenue FROM order_facts)
ON category USING SUM(revenue) GROUP BY region ORDER BY region
""").fetchdf())

## 12. Distinguish a valuable customer from a recently active one

A customer who spent heavily early in the year may now be inactive. Compare recency, frequency, and spending instead of relying on a single total. Keep customers with no purchases in the result; dropping them would distort the customer base and conceal an important segment.
RFM means recency, frequency and monetary value. As of 2025-01-01, we compute days since the last completed purchase, total completed orders, and completed-sales revenue. A left join retains customers with no completed purchases.

These illustrative rule-based segments are not learned labels: never purchased, inactive (>90 days), loyal (>=20 orders), or developing. Business teams should validate thresholds and campaign effectiveness before using them.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE customer_rfm AS
WITH rfm AS (
    SELECT c.customer_id, c.region,
           date_diff('day', MAX(o.order_date), DATE '2025-01-01') AS recency_days,
           COUNT(o.order_id) AS frequency,
           COALESCE(SUM(o.revenue), 0) AS monetary
    FROM customers c LEFT JOIN order_facts o
      ON c.customer_id=o.customer_id AND o.status='completed'
    GROUP BY c.customer_id, c.region
)
SELECT *, CASE WHEN frequency=0 THEN 'never purchased'
               WHEN recency_days > 90 THEN 'inactive'
               WHEN frequency >= 20 THEN 'loyal'
               ELSE 'developing' END AS segment
FROM rfm
""")
display(con.execute("SELECT segment, COUNT(*) AS customers, AVG(monetary) AS avg_revenue FROM customer_rfm GROUP BY segment ORDER BY customers DESC").fetchdf())
assert scalar("SELECT COUNT(*) FROM customer_rfm") == 1200

## 13. Predict a future purchase using only what was known at the time

The marketing team wants a list before the next month begins. Define the decision date first, then separate historical observations from future outcomes. A random split of customer-month rows can mix future patterns into evaluation. Use earlier months to train and later months to evaluate instead.
DuckDB prepares features; scikit-learn trains the model. For each monthly cutoff, only completed orders strictly **before** the cutoff are features. The target is at least one completed purchase in `[cutoff, cutoff + 1 month)`. Features use a trailing 90-day window; labels use the following calendar month, whose length varies.

Train on July–September cutoffs, validate on October, test once on November. Their label windows do not overlap. Customers recur across months because this models future behavior of an existing customer base; it does not measure generalization to unseen customers. Only customers signed up before the cutoff are eligible.

**Critical simplification:** our synthetic statuses are treated as immediately known. Real returned/cancelled outcomes can arrive later; use as-of snapshots and event availability times to avoid leaking future status information. Calendar timestamps here are date-only with no time-zone ambiguity.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE purchase_features AS
WITH cutoffs AS (
    SELECT CAST(d AS DATE) AS cutoff
    FROM generate_series(DATE '2024-07-01', DATE '2024-11-01', INTERVAL '1 month') t(d)
), historical AS (
    SELECT c.customer_id, k.cutoff,
           COUNT(o.order_id) AS orders_90d,
           COALESCE(SUM(o.revenue), 0) AS spend_90d,
           COALESCE(date_diff('day', MAX(o.order_date), k.cutoff), 91) AS recency_days,
           COUNT(DISTINCT o.category) AS categories_90d,
           MAX(o.order_date) AS latest_feature_order
    FROM customers c CROSS JOIN cutoffs k
    LEFT JOIN order_facts o ON o.customer_id=c.customer_id
         AND o.status='completed'
         AND o.order_date >= k.cutoff - INTERVAL '90 days'
         AND o.order_date < k.cutoff
    WHERE c.signup_date < k.cutoff
    GROUP BY c.customer_id, k.cutoff
), labels AS (
    SELECT h.customer_id, h.cutoff, CAST(COUNT(o.order_id)>0 AS INTEGER) AS purchased_next_month
    FROM historical h LEFT JOIN order_facts o
      ON o.customer_id=h.customer_id AND o.status='completed'
      AND o.order_date >= h.cutoff AND o.order_date < h.cutoff + INTERVAL '1 month'
    GROUP BY h.customer_id, h.cutoff
)
SELECT h.*, l.purchased_next_month FROM historical h JOIN labels l USING(customer_id, cutoff)
""")
features = con.execute("SELECT * FROM purchase_features ORDER BY cutoff, customer_id").fetchdf()
assert len(features) == 6000
assert not features.duplicated(["customer_id", "cutoff"]).any()
assert scalar("SELECT COUNT(*) FROM purchase_features WHERE latest_feature_order >= cutoff") == 0
assert features.purchased_next_month.nunique() == 2
# An independent row-level check of the feature and label window boundaries.
sample = features.iloc[0]
source = orders[(orders.customer_id == sample.customer_id) & (orders.status == 'completed') &
                (orders.quantity > 0) & (orders.unit_price > 0)]
history = source[(source.order_date >= sample.cutoff - pd.Timedelta(days=90)) & (source.order_date < sample.cutoff)]
future = source[(source.order_date >= sample.cutoff) & (source.order_date < sample.cutoff + pd.DateOffset(months=1))]
assert sample.orders_90d == len(history)
assert sample.purchased_next_month == int(len(future) > 0)
display(features.head())

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

columns = ["orders_90d", "spend_90d", "recency_days", "categories_90d"]
train = features[features.cutoff < "2024-10-01"]
valid = features[features.cutoff == "2024-10-01"]
test = features[features.cutoff == "2024-11-01"]
assert len(train) == 3600 and len(valid) == len(test) == 1200
# customer_id, cutoff, latest_feature_order and target are intentionally excluded.
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
baseline = DummyClassifier(strategy="prior")
for estimator in (model, baseline):
    estimator.fit(train[columns], train.purchased_next_month)

results = []
for split_name, frame in [("validation", valid), ("test", test)]:
    y = frame.purchased_next_month
    for name, estimator in [("prior baseline", baseline), ("logistic regression", model)]:
        probability = estimator.predict_proba(frame[columns])[:, 1]
        results.append({"split": split_name, "model": name,
                        "prevalence": y.mean(), "roc_auc": roc_auc_score(y, probability),
                        "average_precision": average_precision_score(y, probability),
                        "brier_score": brier_score_loss(y, probability)})
metrics = pd.DataFrame(results)
display(metrics.round(4))
metrics.to_csv(OUT / "model_metrics.csv", index=False)

**Interpretation:** ROC AUC measures ranking (0.5 is the constant-score baseline); average precision summarizes precision/recall and should be compared with class prevalence; Brier score measures probability error (lower is better). The model and its defaults were fixed before inspecting test results. Use validation for any later tuning, and reserve a new untouched test period if you iterate on these results.

A useful score on this synthetic generator is a pipeline demonstration, not evidence of deployment readiness. Real evaluation needs label maturation, drift checks, calibration, costs and benefits, and temporal backtesting. Predictions alone do not establish which customers benefit from a marketing intervention.

## 14. Bring a small business target table into a larger analysis

Regional sales targets arrive as a small Python table. Join those targets to aggregated sales without first transferring every order into pandas. Targets here are arbitrary teaching values: exceeding one does not prove regional effectiveness without accounting for market size and target-setting decisions.
Register a DataFrame explicitly when writing reusable code; implicit Python-variable lookup is convenient but harder to trace. Registration exposes the Python object without persisting it. Use `CREATE TABLE AS SELECT` if you want a stored copy.

The relational API composes operations lazily; fetching or displaying the relation triggers execution. Arrow is useful for typed, columnar exchange, but exporting an entire result still materializes it. Use record batches for large outputs.

In [ ]:
region_targets = pd.DataFrame({"region": ["North", "South", "East", "West"], "target": [900000, 900000, 900000, 900000]})
con.register("region_targets_df", region_targets)
display(con.execute("""
SELECT region, SUM(revenue) AS revenue, MAX(target) AS target,
       100.0 * SUM(revenue)/MAX(target) AS attainment_pct
FROM order_facts JOIN region_targets_df USING(region) GROUP BY region ORDER BY region
""").fetchdf())
con.unregister("region_targets_df")
relation = con.table("order_facts").filter("status = 'completed'").project("order_id, revenue").limit(5)
display(relation.df())
arrow_table = con.execute("SELECT * FROM monthly_report ORDER BY month").fetch_arrow_table()
print(arrow_table.schema)
reader = con.execute("SELECT order_id, revenue FROM order_facts").fetch_record_batch(rows_per_batch=5000)
assert sum(batch.num_rows for batch in reader) == 23930

## 15. Make the analysis usable by the next person

A teammate needs a readable monthly report and typed features for another model. Export CSV for the report and Parquet for the larger typed tables. Verify exported totals against the database; a successful write alone does not prove that the correct rows were exported.
CSV is readable but does not preserve the full schema. Parquet stores typed columns and supports selective reads. Export clean facts, features and reports for other tools.

The output paths below are our own generated paths. SQL path literals are escaped for portability. Use bound values for ordinary queries rather than assembling SQL from user input.

Partitioning by month demonstrates Hive-style directory names. It is useful for larger datasets with frequent month filters, but overkill for this tiny dataset. Partition by a low-cardinality field, not customer_id. We remove only this notebook's previous partition folder on rerun, preventing stale partitions.

In [ ]:
import shutil

def path_literal(path):
    return "'" + str(path).replace("'", "''") + "'"

for table, filename, options in [
    ("monthly_report", "monthly_report.csv", "FORMAT CSV, HEADER"),
    ("order_facts", "order_facts.parquet", "FORMAT PARQUET, COMPRESSION ZSTD"),
    ("purchase_features", "purchase_features.parquet", "FORMAT PARQUET, COMPRESSION ZSTD"),
    ("rejected_orders", "rejected_orders.csv", "FORMAT CSV, HEADER")
]:
    con.execute(f"COPY {table} TO {path_literal(OUT / filename)} ({options})")
partition_dir = OUT / "orders_by_month"
if partition_dir.exists():
    shutil.rmtree(partition_dir)
con.execute(f"""
COPY (SELECT *, strftime(order_date, '%Y-%m') AS order_month FROM order_facts)
TO {path_literal(partition_dir)} (FORMAT PARQUET, PARTITION_BY (order_month), COMPRESSION ZSTD)
""")
pattern = str(partition_dir / "*" / "*.parquet")
display(con.execute("SELECT order_month, SUM(revenue) AS revenue FROM read_parquet(?, hive_partitioning=true) WHERE order_month='2024-11' GROUP BY order_month", [pattern]).fetchdf())
assert con.execute("SELECT COUNT(*) FROM read_parquet(?)", [str(OUT / "order_facts.parquet")]).fetchone()[0] == 23930
assert con.execute("SELECT SUM(revenue) FROM read_parquet(?)", [pattern]).fetchone()[0] == scalar("SELECT SUM(revenue) FROM order_facts")

## 16. Investigate a slow query before changing the hardware

Only November is needed, yet a query seems to scan too much data. Inspect its plan: are filters applied at the scan, are unnecessary columns read, and are joins producing unexpectedly many rows? Measure the actual workload before concluding that another format or more threads will help.
`EXPLAIN` shows the physical plan; `EXPLAIN ANALYZE` executes and measures the query. Look for filters and projected columns in the Parquet scan, and for actual row counts in the analyzed plan. Projection pushdown reads needed columns; filter pushdown can skip row groups when metadata allows it. A tiny one-file example is not a meaningful speed benchmark.

For larger workloads: filter early, aggregate before fetching, avoid many tiny files, inspect join cardinality, and provision local temporary disk for spilling. Not every operator can spill; memory_limit does not cover every allocation or pandas objects. Sorting/windows can be expensive. More threads are not always faster, especially under memory pressure.

In [ ]:
query = "SELECT category, SUM(revenue) AS revenue FROM read_parquet(?) WHERE order_date >= DATE '2024-11-01' GROUP BY category"
for prefix in ["EXPLAIN ", "EXPLAIN ANALYZE "]:
    plan = con.execute(prefix + query, [str(OUT / "order_facts.parquet")]).fetchall()
    print("\n".join(str(row[1]) for row in plan))

## 17. Keep a failed update from leaving a partial result

Suppose a pipeline updates several related tables and fails halfway through. A transaction lets the group of changes succeed or be discarded together. The demonstration deliberately rolls back an update, then checks that the original value remains.
Transactions group changes atomically. A rollback undoes changes within that transaction. If an exception occurs in application code, roll back before reusing the connection. Constraints protect tables at write time; analytical staging still benefits from explicit quality reports.

A macro is reusable SQL expanded into a query. Prefer built-in SQL or a SQL macro for simple transformations; Python UDFs are useful for custom logic but can add overhead and complicate null/type handling.

In [ ]:
con.execute("CREATE OR REPLACE TABLE transaction_demo (id INTEGER PRIMARY KEY, amount DECIMAL(10,2) CHECK(amount >= 0))")
con.execute("INSERT INTO transaction_demo VALUES (1, 10.00)")
con.execute("BEGIN TRANSACTION")
try:
    con.execute("UPDATE transaction_demo SET amount=99 WHERE id=1")
    con.execute("ROLLBACK")  # Deliberately discard this demonstration update.
except Exception:
    con.execute("ROLLBACK")
    raise
assert float(scalar("SELECT amount FROM transaction_demo WHERE id=1")) == 10.0
con.execute("CREATE OR REPLACE MACRO safe_ratio(n, d) AS n / NULLIF(d, 0)")
display(con.execute("SELECT safe_ratio(10,2) AS normal, safe_ratio(10,0) AS undefined").fetchdf())

## 18. Turn query output into a qualified recommendation

Use the results to identify a category for further investigation, not an automatic spending increase. Check margin, return rate, and observation coverage before recommending action. State what the dataset cannot establish: causality, real customer behavior, and campaign profitability.
The following summary derives findings from your actual run. Compare revenue with margin and returns before prioritizing a category. A large revenue month alone does not establish a seasonal effect: this is one synthetic year without a causal design.

Close the write connection, then reopen the database read-only to prove that tables persist. Views depending on registered Python objects would not be portable; our saved views reference only persistent tables.

In [ ]:
best_month = con.execute("SELECT month, revenue FROM monthly_report ORDER BY revenue DESC, month LIMIT 1").fetchone()
best_category = con.execute("SELECT category, SUM(revenue) AS revenue FROM order_facts GROUP BY category ORDER BY revenue DESC, category LIMIT 1").fetchone()
summary = (f"Synthetic retail project\n"
           f"Accepted orders: {scalar('SELECT COUNT(*) FROM clean_orders'):,}\n"
           f"Rejected orders: {scalar('SELECT COUNT(*) FROM rejected_orders')}\n"
           f"Highest-revenue month: {best_month[0]} (${best_month[1]:,.2f})\n"
           f"Highest-revenue category: {best_category[0]} (${best_category[1]:,.2f})\n"
           "Interpretation: teaching data only; no real-world business conclusions.\n")
print(summary)
(OUT / "project_summary.txt").write_text(summary)
con.close()
with duckdb.connect(str(OUT / "retail.duckdb"), read_only=True) as read_con:
    assert read_con.execute("SELECT COUNT(*) FROM purchase_features").fetchone()[0] == 6000
    display(read_con.execute("SELECT * FROM monthly_report ORDER BY month LIMIT 3").fetchdf())
print("Artifacts:", sorted(p.name for p in OUT.iterdir()))

## 19. Exercises
1. Calculate return rates by category. Exclude cancellations from the denominator and show counts alongside percentages.
2. Find each region's highest-revenue category using `DENSE_RANK`. Explain ties.
3. Build weekly revenue with a complete calendar spine. Explain the partial first/last week.
4. Add a 30-day order-count feature without changing the label window. Tune only on validation and use a new future holdout if testing repeatedly.
5. Inject an unknown product key and a malformed date, then confirm quarantine catches both. Update expected counts deliberately.
6. Replace the generated orders with a real permitted dataset. Document currency, grain, event timestamps, return timing, license and missingness before reusing these metrics.

### Hints / example solutions (after reconnecting)
```python
con = duckdb.connect('outputs/duckdb/retail.duckdb')
con.sql("""
  SELECT category,
         COUNT(*) FILTER (WHERE status='returned') AS returned,
         COUNT(*) FILTER (WHERE status IN ('completed','returned')) AS eligible,
         100.0 * returned / NULLIF(eligible, 0) AS return_pct
  FROM order_facts GROUP BY category ORDER BY return_pct DESC
""").show()
con.sql("""
  WITH revenue AS (
    SELECT region, category, SUM(revenue) AS total
    FROM order_facts GROUP BY region, category
  )
  SELECT *, DENSE_RANK() OVER (PARTITION BY region ORDER BY total DESC) AS rank
  FROM revenue QUALIFY rank=1 ORDER BY region, category
""").show()
con.close()
```

## 20. Further topics and official references
This notebook covers the core analytical workflow, not every API. The reference sections below explain operational limits, remote files/extensions, nested data, and troubleshooting.

- [Python API](https://duckdb.org/docs/stable/clients/python/overview)
- [SQL SELECT](https://duckdb.org/docs/stable/sql/statements/select)
- [CSV](https://duckdb.org/docs/stable/data/csv/overview) and [JSON](https://duckdb.org/docs/stable/data/json/overview)
- [Parquet](https://duckdb.org/docs/stable/data/parquet/overview) and [COPY](https://duckdb.org/docs/stable/sql/statements/copy)
- [Window functions](https://duckdb.org/docs/stable/sql/functions/window_functions)
- [Performance tuning](https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads)
- [Concurrency](https://duckdb.org/docs/lts/connect/concurrency)

Documentation may describe newer releases; this notebook prints and records the version actually used.

## 21. Reference: investigating an online store

The preceding walkthrough builds an offline retail dataset, investigates sales and customers, and prepares time-aware features for a repeat-purchase model. These reference notes explain the decisions behind that work for later projects.

The learning style is question-driven: establish the situation, decide what evidence is needed, write the query, check the result, and explain its limits. The goal is to learn how to reason about data as well as how to call a library.

### Get the notebook running

Run from this `Libraries` directory and select the repository's Python environment. The minimal dependencies are:

```sh
python -m pip install duckdb==1.4.3 numpy==1.26.4 pandas==2.3.3 matplotlib==3.9.4 pyarrow==21.0.0 scikit-learn==1.6.1 ipykernel
```

The DuckDB version is a reproducibility pin tested with this repository's Python 3.9 environment, not a latest-version recommendation. Run cells from top to bottom. After installation, the tutorial needs no downloads. It generates its own fictional data using NumPy seed 42. The same dependency versions help reproduce the same outputs.

Outputs go into `outputs/duckdb/`. Re-running replaces this tutorial's generated files, named tables, and partition directory. To reopen the saved data after finishing:

```python
import duckdb
con = duckdb.connect('outputs/duckdb/retail.duckdb', read_only=True)
print(con.execute('SELECT COUNT(*) FROM clean_orders').fetchone())
con.close()
```

### Choose an engine that fits the question

Suppose orders occupy CSV and Parquet files and you need category totals. DuckDB runs SQL within your Python process, avoiding a separate server setup. Use it to scan, join, group and transform data before collecting a small result into Python.

| Need | Useful starting point |
|---|---|
| Local SQL over analytical files | DuckDB |
| Python-centric manipulation of an in-memory table | pandas |
| Expression-based DataFrame pipelines | Polars |
| Embedded transactional application storage | SQLite |
| Central service with many application clients and transactional writes | PostgreSQL |
| Work distributed across machines | A distributed engine such as Spark |

These tools overlap; compare them using your workload, deployment constraints and team's skills. DuckDB does not train a scikit-learn model for you. In this project it produces the feature table, and scikit-learn handles fitting and evaluation.

An in-memory connection (`duckdb.connect()`) lasts only for that database's lifetime. A file-backed connection stores persistent tables. Prefer explicit connections in reusable Python code, rather than relying on a shared global connection. See the [Python API](https://duckdb.org/docs/stable/clients/python/overview).

### Establish the meaning of one row

Before asking which month is best, establish what you are counting. This tutorial has one single-product order per row. A real store may have order headers, line items, shipments, payments and return events. Joining all of them directly can multiply rows and inflate totals.

The generated data has these entities:

| Entity | Size | Meaning |
|---|---:|---|
| Customers | 1,200 | Customer ID, region and signup date |
| Products | 60 | Product ID, category and fixed unit cost |
| Original orders | 24,000 | One product per order, dated during 2024 |
| Raw orders | 24,080 | Original orders plus 80 repeated rows |
| Accepted orders | 23,930 | Distinct orders excluding 70 invalid financial records |

There are also 60 missing channels. These orders remain usable for overall sales and receive the `unknown` channel. Excluding them would change total revenue merely because attribution is incomplete.

Data is entirely synthetic. Currency is fictional USD. The generator omits tax, shipping, partial returns, changing costs and delayed statuses. A model score from this dataset demonstrates mechanics, not real-world predictive power.

### Inspect before importing everything

Start with a schema and a small sample:

```python
con.execute('SELECT * FROM read_csv(?) LIMIT 5', ['orders.csv']).fetchdf()
con.execute('DESCRIBE SELECT * FROM read_parquet(?)', ['products.parquet']).fetchdf()
```

CSV inference is convenient during exploration, but unexpected strings can change the inferred type. Use explicit schemas for established feeds, or preserve raw text and apply `TRY_CAST` to audit conversion failures. Parquet stores types and supports column-oriented access.

You can query files without copying them into persistent tables. Materialize when you need a stable snapshot, repeated transformations, or database constraints. A view stores a query; it is not automatically a stored snapshot of that query's output.

`read_json_auto` is useful for JSON and JSON Lines. Nested arrays require an explicit grain decision. Unnesting three tags produces three rows: counting them counts tags, not the original event. Check [CSV reading](https://duckdb.org/docs/stable/data/csv/overview) and [JSON reading](https://duckdb.org/docs/stable/data/json/overview).

### Preserve uncertainty instead of hiding it

Missing, invalid and zero are different states. An unknown channel can be retained; an invalid price should not silently become zero revenue.

| Expression | Meaning | Analytical consequence |
|---|---|---|
| `TRY_CAST(x AS INTEGER)` | Conversion failure becomes NULL | Count failures before discarding them |
| `x IS NULL` | Test for missingness | `x = NULL` does not serve this purpose |
| `COALESCE(x, 'unknown')` | Substitute a chosen value | Use only when the replacement has a defensible meaning |
| `COUNT(*)` | Count rows | Includes rows with missing attributes |
| `COUNT(x)` | Count non-NULL values | Its denominator may be smaller |
| `NULLIF(denominator, 0)` | Return NULL for zero | Avoid undefined ratios |
| `DECIMAL(18,2)` | Fixed-scale decimal | Useful for money; define when rounding happens |

Keep raw, typed, accepted and rejected layers. The tutorial assigns the first applicable rejection reason; a comprehensive audit might store all failing checks. Exact duplicate removal does not settle conflicting versions of an order. For those, require a source timestamp and a documented version-selection rule.

### Turn a vague request into a measurable quantity

“Sales are improving” needs a definition, comparison period and unit of analysis. In the notebook:

- Order value is quantity × unit price × (1 − discount), rounded per order to cents.
- Completed-sales revenue includes that amount only for completed orders.
- Product cost includes quantity × product unit cost only for completed orders.
- Gross profit is completed-sales revenue minus product cost.
- Average completed order value divides revenue by completed orders.
- Return rate divides returned orders by completed plus returned orders, excluding cancellations.

A different business could reasonably choose a different metric. Returned orders here are excluded from completed revenue, rather than represented as later negative cash flows. This convention must be stated before interpreting the results.

The notebook's order-facts view centralizes those definitions. Test a metric with a tiny example: two completed orders worth $20 and $30, one returned order and one cancellation imply $50 completed revenue, $25 average completed order value, and a 1/3 return rate. They do not imply $12.50 average completed order value or a 1/4 return rate.

### Build queries around the required output

Describe the output before writing SQL: “one row per channel, for completed orders since July, with at least 100 orders.” Then map the definition onto SQL:

```sql
SELECT channel, COUNT(*) AS completed_orders, SUM(quantity) AS units
FROM clean_orders
WHERE status = 'completed' AND order_date >= DATE '2024-07-01'
GROUP BY channel
HAVING COUNT(*) >= 100
ORDER BY completed_orders DESC, channel;
```

`WHERE` limits source rows, `GROUP BY` defines output grain, and `HAVING` filters groups. `ORDER BY` makes display order explicit. `LIMIT` without ordering does not mean “best” or “latest.”

A CTE (`WITH ...`) lets you name intermediate logic. Use it to separate competing ideas, such as monthly aggregation and growth calculations. See [SELECT](https://duckdb.org/docs/stable/sql/statements/select).

For application inputs, bind values:

```python
con.execute('SELECT * FROM clean_orders WHERE channel = ?', [channel]).fetchdf()
```

Placeholders bind values, not arbitrary table names. If identifiers must vary, select them from an allowlist. Do not interpolate untrusted SQL fragments.

### Check a join before trusting its totals

The tutorial joins many orders to one customer and one product. Unique dimension keys are essential. Measure row counts before and after joining and verify that unmatched foreign keys are absent.

Use an inner join when you deliberately require a match. Use a left join when the left population must remain, such as customers with no purchases. A filter on the right table in `WHERE` can discard unmatched rows and unintentionally defeat that left join; place eligibility conditions in `ON` when appropriate.

An anti join finds left-side rows with no match:

```sql
SELECT o.* FROM clean_orders o ANTI JOIN products p USING (product_id);
```

For transactions with multiple line items, aggregate at the intended grain before joining to other one-to-many tables.

### Compare time periods fairly

Monthly revenue needs a calendar spine. Otherwise a missing month disappears and `LAG` compares the current row with the last observed month, not necessarily the previous calendar month.

`LAG` supports previous-period comparisons. Windowed `SUM` supports cumulative totals. `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` covers three rows: it means three months only when one complete row exists per month. First and second moving averages use fewer than three months.

Percentage change is undefined when the previous value is zero. Leave it NULL or document a business convention; do not manufacture an infinite percentage. Compare complete periods, and distinguish missing ingestion from genuine zero activity.

Use `ROW_NUMBER` for an exact top N with a deterministic tie-breaker. Use `DENSE_RANK` to preserve ties. `QUALIFY` filters window results. See [window functions](https://duckdb.org/docs/stable/sql/functions/window_functions).

### Ask what a segment conceals

RFM summarizes recency, frequency and monetary value. The tutorial labels customers with simple rules: never purchased, inactive, loyal or developing. These are teaching thresholds, not validated marketing segments.

A high spender can have low profitability, a recent return, or a long history that makes lifetime spend incomparable to a new customer. Examine membership counts, distributions, customer tenure and stability before operationalizing segments. A segment associated with repeat buying does not prove that a campaign changes buying behavior.

### Set the decision time before building ML features

The model predicts a completed purchase next calendar month. Every feature row has a customer and a cutoff date. Historical orders belong to `[cutoff − 90 days, cutoff)`; labels belong to `[cutoff, cutoff + 1 month)`.

The SQL aggregates history and labels separately so future orders do not multiply historical rows. It checks that the latest feature order precedes the cutoff. A small independent pandas calculation verifies an example's window boundaries.

Training uses July–September snapshots, validation uses October, and testing uses November. Preprocessing is fitted inside a scikit-learn pipeline on training data. Customer ID and dates are excluded from model inputs. Repeated customers across months are intentional: this estimates future performance for existing customers, not cold-start performance for new customers.

Compare logistic regression with a constant-prior baseline. ROC AUC evaluates ranking, average precision should be interpreted alongside class prevalence, and Brier score measures probability error. Do not keep tuning against the test month. If you revise the model after examining it, allocate a new untouched future test period.

Real pipelines must also account for when information became available. An order dated September can receive a return status in October; using the final status to build September features may leak future information. This synthetic tutorial assumes immediate statuses. Real deployment needs historical snapshots or event availability timestamps, matured labels and temporal backtesting.

### Move only the result you need into Python

`fetchone()` returns one row; `fetchall()` returns all rows as tuples; `fetchdf()` creates a pandas DataFrame. Large fetches can exhaust memory even if the SQL engine can execute the query.

Register a small Python table with `con.register('targets', dataframe)` to join it in SQL, then `unregister` it when finished. Registration is not persistence. The relational API provides composable expressions such as `con.table('order_facts').filter(...).project(...)`; fetching executes the relation.

Arrow supports typed exchange. The notebook demonstrates both an Arrow table and record batches. Batch consumption is useful only if downstream code also processes and releases batches rather than accumulating all of them in a list.

### Export something that can be checked independently

The notebook produces:

| Artifact under `outputs/duckdb/` | Purpose |
|---|---|
| `retail.duckdb` | Persistent raw, clean, analytical and feature tables |
| `customers.csv`, `products.parquet`, `orders.csv`, `events.jsonl` | Reproducible generated inputs |
| `rejected_orders.csv` | Audit of invalid orders |
| `monthly_report.csv` | Monthly revenue, profit and window calculations |
| `monthly_revenue.png` | Revenue and moving-average chart |
| `order_facts.parquet` | Typed enriched order dataset |
| `orders_by_month/` | Demonstration of a partitioned Parquet dataset |
| `purchase_features.parquet` | Customer-cutoff features and labels |
| `model_metrics.csv` | Validation/test comparisons against a baseline |
| `project_summary.txt` | Findings calculated from this run |

CSV is accessible but loses schema information. Parquet preserves types and is appropriate for feature handoffs. Partitioning by month can help selective access on larger datasets; too many small partitions add overhead. A small tutorial dataset does not need partitioning for speed.

Verify row counts and money totals after export. The notebook reads Parquet back and checks reconciliation. See [Parquet](https://duckdb.org/docs/stable/data/parquet/overview) and [COPY](https://duckdb.org/docs/stable/sql/statements/copy).

### Diagnose resource use with evidence

Start with `EXPLAIN`, then use `EXPLAIN ANALYZE` to execute and inspect timing and row counts. Look for unexpected join expansion, unnecessary columns and filters applied too late. DuckDB can push column selection and filters into Parquet scans; actual skipped work depends on file metadata and layout.

Avoid declaring one approach faster after one tiny run. Keep data and query semantics fixed, measure repeated trials, distinguish cold and warm runs, record versions and hardware, and verify identical results.

`SET threads = 2` controls parallel execution. `SET memory_limit = '512MB'` controls a database memory budget, not the entire Python process. Some allocations and output objects lie outside it. Larger-than-memory execution can use temporary disk, but not all workloads spill successfully. Configure `temp_directory` and sufficient disk for substantial workloads; avoid fetching the full result when a report will do. Indexes are not a universal cure for large analytical scans. See [performance tuning](https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads).

### Preserve consistency and know the deployment boundary

Use transactions for a group of related writes. On failure, roll back; on success, commit. Constraints such as primary keys and CHECK expressions catch invalid writes, while audit tables explain rejected source data. A SQL macro can centralize a calculation such as division by a nonzero denominator.

For the pinned embedded deployment, organize writes within one process. Multiple processes can open a database read-only when there is no writer; do not treat a local DuckDB file as a general concurrent multi-process write service. Coordinate access or use an appropriate service/catalog architecture. Do not share the module's global connection across threads. See the version-appropriate [concurrency documentation](https://duckdb.org/docs/lts/connect/concurrency).

Remote HTTP/S3 queries and integrations may need extensions, network access and credentials. Extension installation can itself require network access. The tutorial avoids that dependency. Explore `httpfs`, spatial processing and external database integrations only when the data source requires them. Keep credentials outside notebooks and consult the relevant official extension documentation.

Other useful topics beyond this project include LIST/STRUCT/MAP types, UNNEST, ASOF joins for temporal matching, grouping sets, schema evolution with `union_by_name`, database attachment, SQL macros and Python UDFs. Learn them when an analytical question calls for them; a tutorial cannot cover every API or version-specific behavior.

### When a result looks wrong

| Symptom | First investigation |
|---|---|
| Revenue doubled after a join | Check uniqueness of the right-side join key |
| Customers disappeared | Inspect right-table filters after a left join |
| Conversion failed | Preserve the raw field; inspect `TRY_CAST` failures |
| An average is unexpectedly high | Compare `COUNT(*)` with the non-NULL denominator |
| A month vanished | Compare observations with a complete calendar |
| A relative path fails | Start the notebook in this directory |
| Database is locked | Close other writers and stale notebook connections |
| A table is absent after reopening | Was it temporary or only a registered Python object? |
| Memory use grows after a query | Inspect fetch size, Python objects, windows and joins |
| Model metrics are implausibly strong | Audit feature availability and temporal split boundaries |

### Practice the analytical habits

Before each query, write one sentence defining its output grain and population. Predict what should happen to row counts. Name the denominator of every ratio. After execution, reconcile one total independently and state one alternative explanation for the pattern.

For a useful next investigation, compare category revenue with gross margin and return rate. Recommend a category to investigate, explain the evidence, and identify which missing data would change that recommendation. This is a stronger analytical conclusion than simply reporting the largest number.

### Verified execution

All 19 executable cells passed in a fresh kernel on Python 3.9.6 with DuckDB
1.4.3. Checks covered 23,930 accepted orders, 70 rejected orders, uniqueness and
foreign keys, revenue reconciliation, 6,000 customer-cutoff feature rows, a
feature/label boundary example, Parquet round trips, rollback and read-only
reopening of the database. The synthetic model's November test ROC AUC was
approximately 0.7725 versus 0.5 for the constant-prior baseline; this is a
reproducibility observation, not a real-world performance claim.

To repeat the execution using this repository's validator:

```sh
.venv/bin/python scripts/validate_notebooks.py DuckDB.ipynb
```

The validator executes in a temporary directory and retains an executed notebook
under `outputs/validation/`; project data artifacts from that temporary run are
not retained. Run the source notebook normally to keep the project outputs under
`outputs/duckdb/` in this workspace.

## Second project: reconcile a raw ledger without collecting every row into Python

A payment export contains retries, invalid amounts, malformed dates and unknown customer keys. Refunds are signed negative amounts, so deleting every negative amount would inflate net receipts. Reconcile 6,060 raw rows into 60 retries, 50 rejected events and 5,950 usable events. These are generated operational-style records, not actual customer payments. The generator's clean source is used only as a verification oracle, never to fill invalid source fields. Work within a small laptop-sized budget, then identify which intermediate tables or buffers would grow at scale.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through an empty field and a signed refund

`TRY_CAST` distinguishes an unparseable amount from a valid negative amount. `SUM` ignores nulls, so a total can look plausible even when records were lost. Count rejected rows explicitly. Deduplicate exact retries first, then assert event-key uniqueness so a conflicting revision cannot be resolved accidentally.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

p2_rng=np.random.default_rng(91)
p2_n=6000
p2_source=pd.DataFrame({'event_id':np.arange(p2_n),'customer_id':p2_rng.integers(1,121,p2_n),
    'day':pd.Timestamp('2025-01-01')+pd.to_timedelta(p2_rng.integers(0,90,p2_n),unit='D'),
    'amount_cents':p2_rng.integers(100,100000,p2_n)})
p2_source.loc[p2_source.event_id%17==0,'amount_cents'] *= -1 # Signed refunds are valid.
p2_raw=p2_source.astype(str)
p2_raw.loc[0:19,'amount_cents']='bad'
p2_raw.loc[20:29,'amount_cents']=''
p2_raw.loc[30:37,'day']='not-a-date'
p2_raw.loc[38:49,'customer_id']='9999'
p2_raw=pd.concat([p2_raw,p2_raw.iloc[-60:]],ignore_index=True)
p2_customers=pd.DataFrame({'customer_id':np.arange(1,121),'region':np.tile(['East','West','North','South'],30)})
p2_raw.to_csv(p2_out/'ledger_raw.csv',index=False)
# Independent generator-based expected population; never repair a bad value from this oracle.
p2_expected=p2_source.iloc[50:]
p2_expected_total=int(p2_expected.amount_cents.sum())
assert len(p2_raw)==6060 and len(p2_expected)==5950

import duckdb
p2_db=duckdb.connect()
p2_db.execute("SET memory_limit='128MB'")
p2_db.execute('SET threads=2')
p2_db.register('customers_source',p2_customers)
p2_db.execute("CREATE TABLE raw AS SELECT * FROM read_csv(?, all_varchar=true)",[str(p2_out/'ledger_raw.csv')])
p2_db.execute('''
CREATE TABLE typed AS SELECT event_id, TRY_CAST(customer_id AS INTEGER) AS customer_id,
 TRY_CAST(day AS DATE) AS day, TRY_CAST(amount_cents AS BIGINT) AS cents FROM (SELECT DISTINCT * FROM raw)
''')
assert p2_db.sql('SELECT COUNT(*)-COUNT(DISTINCT event_id) FROM typed').fetchone()[0]==0
p2_db.execute('''
CREATE TABLE clean AS SELECT t.* FROM typed t JOIN customers_source c USING(customer_id)
WHERE day IS NOT NULL AND cents IS NOT NULL
''')
assert p2_db.sql('SELECT COUNT(*) FROM typed').fetchone()[0]==6000
assert p2_db.sql('SELECT COUNT(*) FROM clean').fetchone()[0]==5950
assert p2_db.sql('SELECT SUM(cents) FROM clean').fetchone()[0]==p2_expected_total

### Work through the Python boundary

Compare pulling all clean events into pandas and then aggregating with doing the aggregation in SQL before fetching. The SQL path returns four rows. That is a measurable transfer reduction, not proof of a universal runtime improvement. The database's 128 MB memory setting is not a cap on the entire Python process, especially after a DataFrame fetch.

In [ ]:
p2_t=time.perf_counter()
p2_detail=p2_db.sql('SELECT * FROM clean JOIN customers_source USING(customer_id)').df()
p2_python_report=p2_detail.groupby('region').cents.sum().sort_index()
p2_python_seconds=time.perf_counter()-p2_t
p2_t=time.perf_counter()
p2_sql_report=p2_db.sql('SELECT region,SUM(cents)::BIGINT AS net_cents,COUNT(*) AS events FROM clean JOIN customers_source USING(customer_id) GROUP BY region ORDER BY region').df()
p2_sql_seconds=time.perf_counter()-p2_t
assert np.array_equal(p2_python_report.to_numpy(),p2_sql_report.net_cents.to_numpy())
assert p2_sql_report.events.sum()==5950
print(p2_db.sql('EXPLAIN SELECT customer_id,SUM(cents) FROM clean GROUP BY customer_id').fetchone()[1])
p2_sql_report.to_csv(p2_out/'duckdb_payment_report.csv',index=False)
display(p2_sql_report)
print({'detail_rows_transferred':len(p2_detail),'aggregate_rows_transferred':len(p2_sql_report),
       'python_seconds':p2_python_seconds,'sql_seconds':p2_sql_seconds})
p2_db.close()

### Explain the invariant before changing the query

There are 60 redundant retries and 50 unusable events, not 110 unique bad events. The signed total must match the same accepted population across both engines. If the report changes after an optimization, inspect population, duplicate semantics and join cardinality before timing anything.

**Try:** group by customer rather than region. **Worked reasoning:** the number of fetched rows rises to at most 120 but is still much smaller than the event count. For millions of distinct customer IDs, aggregation itself may be memory-intensive; inspect the plan and temporary-disk behavior rather than assuming SQL guarantees constant memory.